# YOLOv8n 1-stage 9-class 재학습

수정된 bbox 데이터로 YOLOv8n을 100 epoch 학습합니다.

- 상세 학습 로그는 파일에 기록됩니다.
- 노트북 출력에는 필요한 최근 로그만 표시합니다.
- 학습은 백그라운드 프로세스로 실행되어 브라우저 연결이 끊겨도 유지됩니다.
- 새 학습은 기존 잘못된 bbox 체크포인트에서 이어받지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

STAGE = '1-stage'
DATA_DIR = Path('/content/drive/MyDrive/TeamProject/test_dataset/1-stage')
DATA_YAML = Path('/content/drive/MyDrive/TeamProject/test_dataset/1-stage/data.yaml')
EXPECTED_NC = 9
EXPECTED_NAMES = ['can_clean', 'can_outer', 'can_inner', 'pet_clean', 'pet_outer', 'pet_inner', 'plastic_clean', 'plastic_outer', 'plastic_inner']

RUNS_DIR = Path("/content/drive/MyDrive/TeamProject/test_dataset/runs")
RUN_NAME = 'yolov8n_1stage_9class_bboxfixed'
RUN_DIR = RUNS_DIR / RUN_NAME

SCRIPT_PATH = Path("/content/drive/MyDrive/TeamProject/test_dataset") / 'train_yolo_1stage_bboxfixed.py'
LOG_PATH = RUNS_DIR / 'yolov8n_1stage_9class_bboxfixed.log'
PID_PATH = RUNS_DIR / 'yolov8n_1stage_9class_bboxfixed.pid'

RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("stage:", STAGE)
print("data:", DATA_DIR)
print("run:", RUN_DIR)
print("log:", LOG_PATH)


## 1. GPU와 패키지 확인


In [ ]:
import sys
import torch
import ultralytics

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 없습니다. 학습을 시작하지 마세요.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM GiB:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
)


## 2. RunPod 경로로 data.yaml 수정 및 검증


In [ ]:
import yaml

if not DATA_YAML.is_file():
    raise FileNotFoundError(DATA_YAML)

with DATA_YAML.open("r", encoding="utf-8") as file:
    data = yaml.safe_load(file)

data["path"] = str(DATA_DIR)
data["train"] = "images/train"
data["val"] = "images/val"
data["nc"] = EXPECTED_NC
data["names"] = EXPECTED_NAMES

with DATA_YAML.open("w", encoding="utf-8") as file:
    yaml.safe_dump(data, file, allow_unicode=True, sort_keys=False)

for split in ["train", "val"]:
    image_dir = DATA_DIR / "images" / split
    label_dir = DATA_DIR / "labels" / split
    image_count = sum(1 for path in image_dir.iterdir() if path.is_file())
    label_count = sum(1 for path in label_dir.glob("*.txt") if path.is_file())
    print(f"{split}: images={image_count}, labels={label_count}")
    if image_count == 0 or image_count != label_count:
        raise RuntimeError(f"{split} 이미지/라벨 수가 비정상입니다.")

with DATA_YAML.open("r", encoding="utf-8") as file:
    checked = yaml.safe_load(file)

assert checked["path"] == str(DATA_DIR)
assert checked["nc"] == EXPECTED_NC
assert checked["names"] == EXPECTED_NAMES

print("\n===== data.yaml =====")
print(DATA_YAML.read_text(encoding="utf-8"))
print("데이터 사전검증 통과")


## 3. 출력 폭주를 막는 학습 스크립트 생성


In [ ]:
trainer_source = r'''
import os
import sys
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO

DATA_YAML = '/content/drive/MyDrive/TeamProject/test_dataset/1-stage/data.yaml'
RUNS_DIR = "/content/drive/MyDrive/TeamProject/test_dataset/runs"
RUN_NAME = 'yolov8n_1stage_9class_bboxfixed'
RUN_DIR = Path(RUNS_DIR) / RUN_NAME
LAST_PT = RUN_DIR / "weights" / "last.pt"
RESUME = os.environ.get("RESUME", "0") == "1"

print("Python:", sys.version.split()[0], flush=True)
print("PyTorch:", torch.__version__, flush=True)
print("Ultralytics:", ultralytics.__version__, flush=True)
print("CUDA:", torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), flush=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 없습니다.")

if RESUME:
    if not LAST_PT.is_file():
        raise FileNotFoundError(f"resume checkpoint 없음: {LAST_PT}")
    print("RESUME:", LAST_PT, flush=True)
    model = YOLO(str(LAST_PT))
    model.train(resume=True)
else:
    if RUN_DIR.exists():
        raise FileExistsError(
            f"기존 결과 폴더가 있습니다: {RUN_DIR}. "
            "새 학습을 덮어쓰지 말고 resume 셀을 사용하세요."
        )

    print("FRESH TRAINING: yolov8n.pt", flush=True)
    model = YOLO("yolov8n.pt")
    model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=32,
        patience=20,
        workers=4,
        cache=False,
        device=0,
        pretrained=True,
        amp=True,
        seed=42,
        deterministic=True,
        verbose=True,
        plots=True,
        save=True,
        save_period=10,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=False,
    )
'''

SCRIPT_PATH.write_text(trainer_source, encoding="utf-8")
print("생성:", SCRIPT_PATH)
print("스크립트 크기:", SCRIPT_PATH.stat().st_size, "bytes")


## 4. 새 100 epoch 학습 시작

이 셀은 학습 프로세스를 실행한 뒤 즉시 제어권을 돌려줍니다. 상세 로그는 로그 파일에 저장됩니다.

동일한 결과 폴더가 이미 있으면 새 학습을 막습니다. 기존 학습을 이어갈 때는 아래 resume 셀을 사용하세요.


In [ ]:
import os
import subprocess
import sys
import time

if RUN_DIR.exists():
    raise RuntimeError(
        f"기존 결과가 있습니다: {RUN_DIR}\n"
        "새 학습을 다시 실행하지 말고 resume 셀을 사용하세요."
    )

log_file = LOG_PATH.open("a", encoding="utf-8")
env = os.environ.copy()
env["RESUME"] = "0"

process = subprocess.Popen(
    [sys.executable, str(SCRIPT_PATH)],
    cwd="/content/drive/MyDrive/TeamProject/test_dataset",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)

PID_PATH.write_text(str(process.pid), encoding="utf-8")
log_file.close()
time.sleep(2)

print("학습 PID:", process.pid)
print("로그:", LOG_PATH)
print("최근 로그 확인 셀을 실행하세요.")


## 5. 실행 상태와 최근 로그만 확인


In [ ]:
import os
from collections import deque

def process_is_running(pid):
    try:
        os.kill(pid, 0)
        return True
    except (ProcessLookupError, PermissionError):
        return False

if PID_PATH.is_file():
    pid = int(PID_PATH.read_text(encoding="utf-8").strip())
    print("PID:", pid)
    print("실행 중:", process_is_running(pid))
else:
    print("PID 파일 없음")

print("\n===== 최근 로그 35줄 =====")
if LOG_PATH.is_file():
    with LOG_PATH.open("r", encoding="utf-8", errors="replace") as file:
        for line in deque(file, maxlen=35):
            print(line, end="")
else:
    print("로그 파일 없음")


## 6. GPU 사용 상태 확인


In [ ]:
import subprocess

print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
).stdout)


## 7. 중단된 학습 재개

Pod 중단이나 마이그레이션 후 `last.pt`가 남아 있을 때만 실행합니다. 실행 중인 학습이 있을 때는 실행하지 마세요.


In [ ]:
import os
import subprocess
import sys
import time

last_pt = RUN_DIR / "weights" / "last.pt"
if not last_pt.is_file():
    raise FileNotFoundError(f"last.pt가 없습니다: {last_pt}")

if PID_PATH.is_file():
    old_pid = int(PID_PATH.read_text(encoding="utf-8").strip())
    try:
        os.kill(old_pid, 0)
        raise RuntimeError(f"기존 학습 PID {old_pid}가 아직 실행 중입니다.")
    except ProcessLookupError:
        pass

log_file = LOG_PATH.open("a", encoding="utf-8")
env = os.environ.copy()
env["RESUME"] = "1"

process = subprocess.Popen(
    [sys.executable, str(SCRIPT_PATH)],
    cwd="/content/drive/MyDrive/TeamProject/test_dataset",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)

PID_PATH.write_text(str(process.pid), encoding="utf-8")
log_file.close()
time.sleep(2)

print("재개 PID:", process.pid)
print("checkpoint:", last_pt)
print("로그:", LOG_PATH)


## 8. 학습 완료 후 결과 요약


In [ ]:
import pandas as pd
from IPython.display import display
from PIL import Image

results_csv = RUN_DIR / "results.csv"
best_pt = RUN_DIR / "weights" / "best.pt"
last_pt = RUN_DIR / "weights" / "last.pt"

print("결과 폴더:", RUN_DIR)
print("best.pt:", best_pt.is_file(), best_pt)
print("last.pt:", last_pt.is_file(), last_pt)

if results_csv.is_file():
    frame = pd.read_csv(results_csv)
    display(frame.tail(10))
else:
    print("results.csv가 아직 없습니다.")

for image_name in [
    "results.png",
    "confusion_matrix_normalized.png",
    "val_batch0_pred.jpg",
]:
    image_path = RUN_DIR / image_name
    if image_path.is_file():
        print("\n", image_name)
        display(Image.open(image_path))


In [ ]:
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/TeamProject/test_dataset/runs/"
    "yolov8n_1stage_9class_bboxfixed"
)

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
DATA_YAML = Path("/content/drive/MyDrive/TeamProject/test_dataset/1-stage/data.yaml")

print("RUN_DIR:", RUN_DIR)
print("best.pt:", BEST_PT.is_file(), BEST_PT)
print("last.pt:", LAST_PT.is_file(), LAST_PT)
print("results.csv:", RESULTS_CSV.is_file())
print("data.yaml:", DATA_YAML.is_file())

In [ ]:
from ultralytics import YOLO

model = YOLO(str(BEST_PT))

metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=32,
    device=0,
    workers=4,
    plots=True,
    verbose=False,
    project="/content/drive/MyDrive/TeamProject/test_dataset/runs",
    name="yolov8n_1stage_9class_bboxfixed_val_best",
    exist_ok=True,
)

print("\n===== 1-stage best.pt 전체 성능 =====")
print(f"Precision:     {metrics.box.mp:.5f}")
print(f"Recall:        {metrics.box.mr:.5f}")
print(f"전체 mAP50:    {metrics.box.map50:.5f}")
print(f"전체 mAP50-95: {metrics.box.map:.5f}")

In [ ]:
class_names = model.names

class_map50 = metrics.box.ap50
class_map50_95 = metrics.box.maps

print("===== 1-stage 클래스별 성능 =====")

for class_id, class_name in class_names.items():
    print(
        f"{class_id} {class_name:<15} "
        f"mAP50: {float(class_map50[class_id]):.4f} | "
        f"mAP50-95: {float(class_map50_95[class_id]):.4f}"
    )

In [ ]:
import pandas as pd

df = pd.read_csv(RESULTS_CSV)

# Ultralytics CSV 열 이름 앞뒤에 공백이 있을 수 있음
df.columns = df.columns.str.strip()

required_columns = [
    "epoch",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"results.csv에 필요한 열이 없습니다: {missing_columns}"
    )

best_map50_row = df.loc[
    df["metrics/mAP50(B)"].idxmax()
]

best_map50_95_row = df.loc[
    df["metrics/mAP50-95(B)"].idxmax()
]

# Ultralytics 탐지 성능을 비교할 때 쓰는 근사 fitness
df["fitness_proxy"] = (
    0.1 * df["metrics/mAP50(B)"]
    + 0.9 * df["metrics/mAP50-95(B)"]
)

best_fitness_row = df.loc[
    df["fitness_proxy"].idxmax()
]

print("===== 최고 mAP50 =====")
print(f"epoch: {int(best_map50_row['epoch']) + 1}")
print(
    f"mAP50: "
    f"{best_map50_row['metrics/mAP50(B)']:.5f}"
)
print(
    f"mAP50-95: "
    f"{best_map50_row['metrics/mAP50-95(B)']:.5f}"
)

print("\n===== 최고 mAP50-95 =====")
print(f"epoch: {int(best_map50_95_row['epoch']) + 1}")
print(
    f"precision: "
    f"{best_map50_95_row['metrics/precision(B)']:.5f}"
)
print(
    f"recall: "
    f"{best_map50_95_row['metrics/recall(B)']:.5f}"
)
print(
    f"mAP50: "
    f"{best_map50_95_row['metrics/mAP50(B)']:.5f}"
)
print(
    f"mAP50-95: "
    f"{best_map50_95_row['metrics/mAP50-95(B)']:.5f}"
)

print("\n===== 최고 fitness 근사값 =====")
print(f"epoch: {int(best_fitness_row['epoch']) + 1}")
print(f"fitness: {best_fitness_row['fitness_proxy']:.6f}")

In [ ]:
import json

summary = {
    "model": str(BEST_PT),
    "data": str(DATA_YAML),
    "overall": {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    },
    "per_class": {
        class_names[class_id]: {
            "mAP50": float(class_map50[class_id]),
            "mAP50_95": float(class_map50_95[class_id]),
        }
        for class_id in class_names
    },
    "best_epoch_by_map50_95": {
        "epoch": int(best_map50_95_row["epoch"]) + 1,
        "precision": float(
            best_map50_95_row["metrics/precision(B)"]
        ),
        "recall": float(
            best_map50_95_row["metrics/recall(B)"]
        ),
        "mAP50": float(
            best_map50_95_row["metrics/mAP50(B)"]
        ),
        "mAP50_95": float(
            best_map50_95_row["metrics/mAP50-95(B)"]
        ),
    },
}

SUMMARY_PATH = (
    RUN_DIR / "best_model_evaluation_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\n저장 완료:", SUMMARY_PATH)